# C2 — Can binding be learned *within* one domain? (Stages 3b and 3c)

**Question.** Remove the transfer problem: train and test inside the same domain, holding out
whole sequence **positions** so the model cannot interpolate from other substitutions at a position
it has seen. Is binding predictable then — and is a single fit trustworthy?

**What it does.** 7-fold cross-validation grouped by position, an inner validation split also
held out by position, a variance-floored scaler, a whitened 64-component PCA, three model seeds
per fold. Run **six times per domain** (Stage 3c): a single fit — what Stage 3b reported — cannot
show how much the answer depends on the random split.

**Two partition modes, run side by side.** Re-checking the original code showed that Stages
3b–3d split positions into folds with scikit-learn's `GroupKFold`, which assigns groups
deterministically and does not shuffle by default. Every "repeat" therefore reused the same
partition; only the inner validation split and the model seeds changed.
- `original` — the unshuffled procedure, reproducing the recorded numbers exactly.
- `corrected` — positions shuffled with the seed before assignment (the Stage 3e design), so each
  repeat is a genuinely different partition.
Set `MODES` below to run one or both. The results table lists them side by side.

**Decision written for B3:** `binding_decision.json` records whether the trained head beats the
zero-shot ESM-2 score on GRB2-SH3 in any mode. If it does not, the atlas binding axis stays
rank-matched ESM-2.

**Inputs:** A2 features. **Runtime:** about 15 min (both modes), T4 GPU.
**Outputs:** `within_domain_repeats.csv`, `within_domain_summary.csv`, `binding_decision.json`, `RESULTS_C2.md`.
Explanation: `docs/C2_binding_within_domain.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'C2', 'Binding within one domain', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

In [ ]:
MODES = ['original', 'corrected']   # either or both

DD = pd.read_parquet(R.res('A2', 'features_ddpca.parquet'))
EMB = C.load_embeddings(R.res('A2', 'esm_domain_embeddings.npz'))
Xd, DB = C.build_X(DD, EMB)
B = DB[DB.ddG_bind.notna() & DB.binding_reliable.eq(True)].copy()
XB = Xd[B.index.values].astype(np.float32); B = B.reset_index(drop=True)
print(B.groupby('domain_key').agg(measurements=('pos', 'size'), positions=('pos', 'nunique')).to_string())

In [ ]:
if R.done('within_domain_repeats.csv'):
    RP = pd.read_csv(R.out('within_domain_repeats.csv')); CI = pd.read_csv(R.out('single_fit_ci.csv'))
    print('loaded from this folder')
else:
    rows, ci = [], []; t0 = time.time()
    for mode in MODES:
        for dom in ['GB1', 'GRB2-SH3', 'PSD95-PDZ3']:
            d = B[B.domain_key == dom]
            for s in range(6):
                r = C.eval_budget(B, XB, dom, d.pos.nunique(), len(d), seed=s, n_splits=7, model_seeds=3, epochs=60,
                                  mode=mode, keep_oof=(s == 0))
                if r is None: continue
                if s == 0:          # seed 0 is the single fit Stage 3b reported
                    lo, hi = C.boot_ci(r.pop('_y'), r.pop('_oof'), 1000); ci.append(dict(mode=mode, domain=dom, lo=lo, hi=hi))
                rows.append(r)
            print(f'  {mode:<9} {dom:<11} {(time.time() - t0) / 60:.1f} min')
    RP, CI = pd.DataFrame(rows), pd.DataFrame(ci)
    RP.to_csv(R.out('within_domain_repeats.csv'), index=False); CI.to_csv(R.out('single_fit_ci.csv'), index=False)
SUM = RP.groupby(['mode', 'domain']).agg(n=('n', 'first'), positions=('n_pos', 'first'), repeats=('rho', 'size'),
                                         repeats_median=('rho', 'median'), repeats_lo=('rho', 'min'), repeats_hi=('rho', 'max'),
                                         esm=('esm', 'first')).reset_index()
SUM = SUM.merge(RP[RP.seed == 0][['mode', 'domain', 'rho']].rename(columns={'rho': 'single_fit_rho'}), on=['mode', 'domain'])
SUM = SUM.merge(CI, on=['mode', 'domain']).rename(columns={'lo': 'single_fit_ci_lo', 'hi': 'single_fit_ci_hi'})
SUM['spread'] = SUM.repeats_hi - SUM.repeats_lo; SUM['crosses_zero'] = (SUM.repeats_lo < 0) & (SUM.repeats_hi > 0)
SUM['beats_esm'] = SUM.repeats_median > -SUM.esm   # signed: a wrong-direction correlation is not a win
SUM.to_csv(R.out('within_domain_summary.csv'), index=False); print(SUM.round(3).to_string(index=False))

In [ ]:
g = SUM[SUM.domain == 'GRB2-SH3']
DEC = dict(head_beats_esm=bool(g.beats_esm.any()),
           grb2_median_rho={m: float(v) for m, v in zip(g['mode'], g.repeats_median)},
           grb2_esm_rho=float(g.esm.iloc[0]),
           predictor_of_record='trained head' if g.beats_esm.any() else 'esm_zero_shot')
json.dump(DEC, open(R.out('binding_decision.json'), 'w'), indent=1); print(DEC)
for _, r in SUM.iterrows():
    R.metric(f'{r["mode"]}.{r.domain}.single_fit_rho', r.single_fit_rho)
    for c in ('n', 'positions', 'repeats_median', 'repeats_lo', 'repeats_hi', 'spread'):
        R.metric(f'{r["mode"]}.{r.domain}.{c}', r[c])
    R.metric(f'{r.domain}.esm', r.esm)
R.metric('head_beats_esm_on_grb2', DEC['head_beats_esm'])
R.section('Within-domain binding, six repeats per domain', 'Spearman ρ on held-out positions. single_fit = seed 0 (what '
          'Stage 3b reported), with its bootstrap CI; the repeats show what the bootstrap cannot: variance of the procedure.', SUM, '{:.3f}')
R.section('Decision for the atlas (read by B3)', '```\n' + json.dumps(DEC, indent=1) + '\n```')
o = SUM.set_index(['mode', 'domain'])
R.finish([f"PSD95-PDZ3 (84 positions): median ρ {o.loc[('original','PSD95-PDZ3'),'repeats_median']:+.3f} over six repeats — learnable."
          if 'original' in SUM['mode'].values else 'PSD95-PDZ3 learnable.',
          f"GRB2-SH3 (56 positions): median ρ " + ', '.join(f"{o.loc[(m,'GRB2-SH3'),'repeats_median']:+.3f} ({m})" for m in SUM['mode'].unique())
          + f"; the six repeats cross zero, and ESM gives {DEC['grb2_esm_rho']:+.3f}.",
          f"Binding predictor of record: {DEC['predictor_of_record']}.",
          'The single +0.124 fit reported by Stage 3b is one draw from this spread, not a result.'])